# DeepGraph-GBM — demo inference on one Visium slide

Loads a trained checkpoint and runs end-to-end inference on one held-out GBM section:
predicted niche map, MES probability map, and slide-level survival risk score.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch

from deepgraph_gbm.models.multitask import DeepGraphGBM
from deepgraph_gbm.training.evaluate import NICHE_CLASSES
from deepgraph_gbm.interpret.spatial_maps import plot_niche_map, plot_mes_map

REPO = Path('..').resolve()
cfg = __import__('yaml').safe_load(open(REPO / 'configs/default.yaml'))
splits = json.loads((REPO / 'configs/splits.json').read_text())
print('test patients:', splits['test'])

In [ ]:
# load graphs + pick one test section
data_list = torch.load(REPO / 'data/processed/graphs.pt', weights_only=False)
test = [d for d in data_list if d.patient in splits['test']]
d = test[0]
print(f'section {d.section} | patient {d.patient} | {d.n_spots} spots | {d.edge_index.shape[1]} edges')

In [ ]:
# load model
model = DeepGraphGBM(in_dim=d.x.shape[1], hidden_dims=cfg['model']['hidden_dims'], dropout=cfg['model']['dropout'])
ckpt = torch.load(REPO / 'models/seed42/best_model.pt', weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()

with torch.no_grad():
    out = model(d.x, d.edge_index)
prob = torch.softmax(out['niche_logits'], 1)
pred = prob.argmax(1)
print('slide risk score:', round(out['risk'].item(), 3))
print('predicted niche fractions:', pd.Series(pred.numpy()).map(lambda i: NICHE_CLASSES[i]).value_counts(normalize=True).round(3).to_dict())

In [ ]:
# spatial maps (coordinates are stored during graph build; reload from processed meta if needed)
import numpy as np
coords = np.load(REPO / 'data/processed' / f'{d.section}_coords.npy')  # saved by 02_build_graphs.py
labels = pd.Series(pred.numpy()).map(lambda i: NICHE_CLASSES[i])
plot_niche_map(coords, labels, f'{d.section} — predicted niches', str(REPO / 'results/figures/demo_niche_map.png'))
plot_mes_map(coords, out['mes_prob'].numpy(), f'{d.section} — MES probability', str(REPO / 'results/figures/demo_mes_map.png'))
print('saved figures to results/figures/')